In [ ]:
import sys
import asyncio
import nest_asyncio

# Force Windows to use the Proactor Event Loop BEFORE anything else runs
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Patch Jupyter's loop
nest_asyncio.apply()

print("Event loop policy set successfully!")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PlaywrightURLLoader
from langchain_community.document_transformers import BeautifulSoupTransformer

### Setup

In [ ]:
target_urls = [
    "https://qdrant.tech/documentation/quick-start/",
    "https://qdrant.tech/documentation/concepts/collections/",
    "https://qdrant.tech/documentation/manage-data/points/",
    "https://qdrant.tech/documentation/manage-data/vectors/",
    "https://docs.cohere.com/docs/the-cohere-platform",
    "https://docs.cohere.com/docs/get-started-installation",
    "https://docs.cohere.com/docs/cohere-embed"
]

In [ ]:
loader = PlaywrightURLLoader(
    urls=target_urls,
    remove_selectors=["header", "footer", "nav"] 
)

# Load documents asynchronously
raw_docs = await loader.aload()

bs_transformer = BeautifulSoupTransformer()
docs = bs_transformer.transform_documents(
    raw_docs,
    tags_to_extract=["p", "h1", "h2", "h3", "li", "code"]
)

for doc in docs:
    print(f"{doc.metadata['source']} → {len(doc.page_content)} chars")

In [1]:
import requests
from langchain_core.documents import Document

target_urls = [
    "https://qdrant.tech/documentation/quick-start/",
    "https://qdrant.tech/documentation/concepts/collections/",
    "https://qdrant.tech/documentation/manage-data/points/",
    "https://qdrant.tech/documentation/manage-data/vectors/",
    "https://docs.cohere.com/docs/the-cohere-platform",
    "https://docs.cohere.com/docs/get-started-installation",
    "https://docs.cohere.com/docs/cohere-embed"
]

docs = []

for url in target_urls:
    print(f"Fetching {url}...")
    
    # 1. Prepend the Jina Reader API to the target URL
    jina_url = f"https://r.jina.ai/{url}"
    
    # 2. Add a simple header (Jina asks for this just to be polite)
    headers = {"Accept": "application/json"}
    
    # 3. Make the request
    response = requests.get(jina_url, headers=headers)
    
    if response.status_code == 200:
        # Jina returns JSON containing the clean Markdown in the 'data.content' field
        data = response.json().get("data", {})
        markdown_text = data.get("content", "")
        
        # 4. Create standard LangChain Documents!
        doc = Document(page_content=markdown_text, metadata={"source": url})
        docs.append(doc)
        
        print(f"  → Success! Length: {len(markdown_text)} chars")
    else:
        print(f"  → Failed with status code: {response.status_code}")

print("\nAll done!")

Fetching https://qdrant.tech/documentation/quick-start/...
  → Success! Length: 16389 chars
Fetching https://qdrant.tech/documentation/concepts/collections/...
  → Success! Length: 0 chars
Fetching https://qdrant.tech/documentation/manage-data/points/...
  → Success! Length: 23761 chars
Fetching https://qdrant.tech/documentation/manage-data/vectors/...
  → Success! Length: 13692 chars
Fetching https://docs.cohere.com/docs/the-cohere-platform...
  → Success! Length: 8997 chars
Fetching https://docs.cohere.com/docs/get-started-installation...
  → Success! Length: 8059 chars
Fetching https://docs.cohere.com/docs/cohere-embed...
  → Success! Length: 4588 chars

All done!
